# 모델링과 모든 작업 중 문제가 생겨 통합 전처리 준비
기준 : 설립구분, 지역, 대계열 확장을 마친  중계 탈락율 파일을 기준으로 모든 연도에 존재하는 공통 조합, 학교만을 남기고 그것을 기준으로 나머지 피처들을 채움.



In [ ]:
# 1. 라이브러리 불러오기
import pandas as pd

In [ ]:
# 2. 파일 불러오기
file_path = '/content/2014~2023_중도탈락_공통조합_전처리완료_v2.csv'
df = pd.read_csv(file_path)
df.head()

In [ ]:
# 3. 공통 조합만 필터링한 데이터 확인
print(f'총 행 수: {len(df)}')
print(f'기준연도 범위: {df["기준연도"].min()} ~ {df["기준연도"].max()}')
print(f'학교 수: {df["학교명"].nunique()}')
print(f'계열 수: {df["계열"].nunique()}')

In [ ]:
# 4. 필요시 저장
df.to_csv('/content/전처리된_중도탈락_공통조합.csv', index=False, encoding='utf-8-sig')

In [ ]:
#신입생 충원율 관련 피처들 추가
middle_path = '/content/2014~2023_중도탈락_공통조합_전처리완료_v2.csv' # 자신 기준으로 맞출 것
fresh_path = '/content/신입생_충원_현황(2014~2023).csv'  # 자신 기준으로 맞출 것
df_mid = pd.read_csv(middle_path)
df_fresh = pd.read_csv(fresh_path)
df_mid.head()

#  컬럼명 통일 및 병합
df_mid = df_mid.rename(columns={'기준연도': '기준년도', '계열': '대계열'})
df_fresh = df_fresh.rename(columns={'학교': '학교명'})
cols = ['기준년도', '학교명', '대계열', '충원율(%)', '경쟁률', '충원율_평균', '충원율_표준편차', '충원안정성비율']
df_fresh_selected = df_fresh[cols]
df_merged = pd.merge(df_mid, df_fresh_selected, on=['기준년도', '학교명', '대계열'], how='left')
df_merged.head()

# 4. 저장
df_merged.to_csv('/content/2014~2023_중도탈락_공통조합_충원정보포함.csv', index=False, encoding='utf-8-sig')

In [ ]:
#전임교원 관련 피처들 추가
base_path = '/content/2014~2023_중도탈락_공통조합_충원정보포함.csv' # 자신 기준으로 맞출 것
faculty_path = '/content/학과별_전임교원_최종.csv'  # 자신 기준으로 맞출 것
df_base = pd.read_csv(base_path)
df_faculty = pd.read_csv(faculty_path)
df_base.head()

# 전임교원 지표 컬럼 정리 및 계산
df_faculty['전임교원1인당재학생_계산'] = df_faculty['재학생'] / df_faculty['전임교원']
df_faculty = df_faculty.rename(columns={'기준연도': '기준년도'})
faculty_cols = ['기준년도', '학교명', '대계열', '전임교원1인당학생정원', '전입교원1인당재학생', '전임교원1인당재학생_계산']
df_faculty_selected = df_faculty[faculty_cols]

# 4. 병합 및 저장
df_final = pd.merge(df_base, df_faculty_selected, on=['기준년도', '학교명', '대계열'], how='left')
df_final.to_csv('/content/2014~2023_중도탈락_충원+전임교원_포함.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
#재학생 충원율 관련 피처 추가
base_path = '/content/2014~2023_중도탈락_충원_전임교원_포함.csv'  # 기존 병합된 파일
retention_path = '/content/학과별_재학생충원율_최종임.csv'  # 새로운 재학생 충원율 파일
df_base = pd.read_csv(base_path)
df_retention = pd.read_csv(retention_path)

#  지역 정보 병합
df_base_region = df_base[['기준년도', '학교명', '대계열', '지역']].drop_duplicates()
df_retention = df_retention.rename(columns={'기준연도': '기준년도'})
df_retention = pd.merge(df_retention, df_base_region, on=['기준년도', '학교명', '대계열'], how='left')

#지역 기준 통계 계산 및 파생 변수 생성
stats = df_retention.groupby(['기준년도', '지역'])['재학생충원율'].agg(['mean', 'std']).reset_index()
stats = stats.rename(columns={'mean': '재학생충원율_평균_지역기준', 'std': '재학생충원율_표준편차_지역기준'})
df_retention = pd.merge(df_retention, stats, on=['기준년도', '지역'], how='left')
df_retention['재학생충원율_편차_지역기준'] = df_retention['재학생충원율'] - df_retention['재학생충원율_평균_지역기준']
df_retention['재학생충원안정성비율_지역기준'] = abs(df_retention['재학생충원율_편차_지역기준']) / df_retention['재학생충원율_표준편차_지역기준']

#학교+대계열 기준 평균 정리 후 병합
df_retention_grouped = df_retention.groupby(['기준년도', '학교명', '대계열']).agg({
    '재학생충원율_평균_지역기준': 'mean',
    '재학생충원율_편차_지역기준': 'mean',
    '재학생충원율_표준편차_지역기준': 'mean',
    '재학생충원안정성비율_지역기준': 'mean'
}).reset_index()
df_final = pd.merge(df_base, df_retention_grouped, on=['기준년도', '학교명', '대계열'], how='left')

# 저장
df_final.to_csv('/content/2014~2023_중도탈락_충원_전임교원_재학생충원율포함_최종.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
# 취업률 관련 피처들 통합
base_path = '/content/2014~2023_중도탈락_충원_전임교원_재학생충원율포함_최종.csv'
career_path = '/content/2014_2023_계열통합_진로지표_전체통합.xlsx'
df_base = pd.read_csv(base_path)
df_career = pd.read_excel(career_path)
df_base.head()

# 컬럼명 정리 및 병합 준비
df_career = df_career.rename(columns={'계열': '대계열'})
career_cols = ['기준년도', '학교명', '대계열', '취업률(%)', '진로진출률(%)', '졸업자대비진로성취(%)']
df_career = df_career[career_cols]

# 병합 및 저장
df_final = pd.merge(df_base, df_career, on=['기준년도', '학교명', '대계열'], how='left')
df_final.to_csv('/content/2014~2023_중도탈락_모든지표포함_최종.csv', index=False, encoding='utf-8-sig')
df_final.head()


In [ ]:
# 재학생 1인당 전임교원 피처 만들어서 추가
base_path = '/content/2014~2023_중도탈락_모든지표포함_최종.csv'
faculty_path = '/content/학과별_전임교원_최종.csv'
df_base = pd.read_csv(base_path)
df_faculty = pd.read_csv(faculty_path)

#  재학생1인당 전임교원 비율 계산
df_faculty = df_faculty.rename(columns={'기준연도': '기준년도'})
df_faculty['재학생1인당전임교원'] = df_faculty['전임교원'] / df_faculty['재학생']
df_faculty = df_faculty[['기준년도', '학교명', '대계열', '재학생1인당전임교원']]

# 병합 및 저장
df_final = pd.merge(df_base, df_faculty, on=['기준년도', '학교명', '대계열'], how='left')
df_final.to_csv('/content/2014~2023_중도탈락_모든지표포함_재학생1인당전임교원추가.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
# 1인당 장학금 피처 추가
base_path = '/content/2014~2023_중도탈락_모든지표포함_재학생1인당전임교원추가.csv'
scholarship_path = '/content/학과별_장학금_최종.csv'
df_base = pd.read_csv(base_path)
df_sch = pd.read_csv(scholarship_path)

# 컬럼명 통일 및 병합 준비
df_sch = df_sch.rename(columns={'기준연도': '기준년도'})
df_sch = df_sch[['기준년도', '학교명', '대계열', '1인당장학금']]

#  병합 및 저장
df_final = pd.merge(df_base, df_sch, on=['기준년도', '학교명', '대계열'], how='left')
df_final.to_csv('/content/2014~2023_중도탈락_모든지표포함_장학금추가.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
# 진학률 피처 추가
base_path = '/content/2014~2023_중도탈락_모든지표포함_장학금추가.csv'
adv_path = '/content/2014~2023_졸업생_진학률_계열통합_완성.xlsx'
df_base = pd.read_csv(base_path)
df_adv = pd.read_excel(adv_path)

# 컬럼명 정리 및 병합 준비
df_adv = df_adv.rename(columns={'기준연도': '기준년도', '학과': '대계열'})
df_adv = df_adv[['기준년도', '학교명', '대계열', '진학률(%)']]

#  병합 및 저장
df_final = pd.merge(df_base, df_adv, on=['기준년도', '학교명', '대계열'], how='left')
df_final.to_csv('/content/2014~2023_중도탈락_모든지표포함_진학률추가.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
# 등록금 피처 추가
base_path = '/content/2014~2023_중도탈락_모든지표포함_진학률추가.csv'
tuition_path = '/content/학과별_등록금_최종.csv'
df_base = pd.read_csv(base_path)
df_tuition = pd.read_csv(tuition_path)

# 컬럼 정리 및 병합 준비
df_tuition = df_tuition.rename(columns={'기준연도': '기준년도'})
df_tuition = df_tuition[['기준년도', '학교명', '대계열', '등록금']]

# 4병합 및 저장
df_final = pd.merge(df_base, df_tuition, on=['기준년도', '학교명', '대계열'], how='left')
df_final.to_csv('/content/2014~2023_중도탈락_모든지표포함_등록금추가.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
# 수도권/비수도권 구분
# 파일 불러오기
path = '/content/2014~2023_중도탈락_모든지표포함_등록금추가.csv'
df = pd.read_csv(path)

#  수도권 여부 컬럼 생성
metro_regions = ['서울', '경기', '인천']
df['수도권여부'] = df['지역'].apply(lambda x: '수도권' if str(x).strip() in metro_regions else '비수도권')

# 저장
df.to_csv('/content/2014~2023_중도탈락_모든지표포함_수도권구분추가.csv', index=False, encoding='utf-8-sig')
df[['학교명', '지역', '수도권여부']].drop_duplicates().head()

In [ ]:
#신입생 충원율, 수도권여부를 이용하여 파생피처 생성
fresh_path = '/content/신입생_충원_현황.csv'  # 신입생 충원율 데이터 (학교명, 대계열 포함)
full_path = '/content/2014~2023_중도탈락_모든지표포함_수도권구분추가.csv'  # 수도권여부 포함된 통합 데이터
df_fresh = pd.read_csv(fresh_path)
df_full = pd.read_csv(full_path)

# 신입생 충원율 데이터 정리 및 수도권 여부 병합
df_fresh = df_fresh.rename(columns={'학교': '학교명'})
df_fresh_meta = df_fresh[['기준년도', '학교명', '대계열', '충원율(%)']]
df_fresh_meta = pd.merge(df_fresh_meta, df_full[['기준년도', '학교명', '대계열', '수도권여부']],
                         on=['기준년도', '학교명', '대계열'], how='left')

# 수도권 여부 기준 통계량 계산
# - 평균: 해당 연도 내 수도권/비수도권 그룹의 평균 충원율
# - 표준편차: 해당 그룹의 충원율 분산 정도
stats = df_fresh_meta.groupby(['기준년도', '수도권여부'])['충원율(%)'].agg(['mean', 'std']).reset_index()
stats = stats.rename(columns={
    'mean': '신입생_충원율_평균_수도권여부기준',
    'std': '신입생_충원율_표준편차_수도권여부기준'
})

# 편차 및 안정성 비율 계산
# - 편차: 해당 학교의 충원율 - 수도권/비수도권 평균 충원율
# - 안정성비율: 편차의 절대값 ÷ 표준편차 → 클수록 평균에서 멀리 있음
df_result = pd.merge(df_fresh_meta, stats, on=['기준년도', '수도권여부'], how='left')
df_result['신입생_충원율_편차_수도권여부기준'] = df_result['충원율(%)'] - df_result['신입생_충원율_평균_수도권여부기준']
df_result['신입생_충원안정성비율_수도권여부기준'] = abs(df_result['신입생_충원율_편차_수도권여부기준']) / df_result['신입생_충원율_표준편차_수도권여부기준']

# 최종 통합 파일 병합 및 저장
df_final_merge = df_result[[
    '기준년도', '학교명', '대계열',
    '신입생_충원율_평균_수도권여부기준',
    '신입생_충원율_편차_수도권여부기준',
    '신입생_충원율_표준편차_수도권여부기준',
    '신입생_충원안정성비율_수도권여부기준']]
df_full_final = pd.merge(df_full, df_final_merge, on=['기준년도', '학교명', '대계열'], how='left')
df_full_final.to_csv('/content/2014~2023_중도탈락_수도권기준_신입충원안정성추가.csv', index=False, encoding='utf-8-sig')
df_full_final.head()

In [ ]:
# 파생 피처 추가
df_base = pd.read_csv('/content/2014~2023_중도탈락_수도권기준_신입충원안정성추가.csv')
df_retention = pd.read_csv('/content/학과별_재학생충원율_최종임.csv')

#  재학생 충원율에 지역, 설립구분, 수도권 정보 병합
df_retention = df_retention.rename(columns={'기준연도': '기준년도'})
merge_info = df_base[['기준년도', '학교명', '대계열', '지역', '설립구분', '수도권여부']].drop_duplicates()
df_retention = pd.merge(df_retention, merge_info, on=['기준년도', '학교명', '대계열'], how='left')

# 재학생 충원율 기준 지역기준 파생 피처 계산
g_cols = ['기준년도', '지역', '설립구분', '대계열']
df_retention['재학생충원율_평균_지역기준'] = df_retention.groupby(g_cols)['재학생충원율'].transform('mean')
df_retention['재학생충원율_표준편차_지역기준'] = df_retention.groupby(g_cols)['재학생충원율'].transform('std')
df_retention['재학생충원율_편차_지역기준'] = df_retention['재학생충원율'] - df_retention['재학생충원율_평균_지역기준']
df_retention['재학생충원안정성비율_지역기준'] = abs(df_retention['재학생충원율_편차_지역기준']) / df_retention['재학생충원율_표준편차_지역기준']

# 수도권 여부 기준 파생 피처 계산
m_cols = ['기준년도', '수도권여부']
df_retention['재학생충원율_평균_수도권여부기준'] = df_retention.groupby(m_cols)['재학생충원율'].transform('mean')
df_retention['재학생충원율_표준편차_수도권여부기준'] = df_retention.groupby(m_cols)['재학생충원율'].transform('std')
df_retention['재학생충원율_편차_수도권여부기준'] = df_retention['재학생충원율'] - df_retention['재학생충원율_평균_수도권여부기준']
df_retention['재학생충원안정성비율_수도권여부기준'] = abs(df_retention['재학생충원율_편차_수도권여부기준']) / df_retention['재학생충원율_표준편차_수도권여부기준']

#  피처 추출 및 base 파일에 병합
merge_cols = [
    '기준년도', '학교명', '대계열', '재학생충원율',
    '재학생충원율_평균_지역기준', '재학생충원율_편차_지역기준', '재학생충원율_표준편차_지역기준', '재학생충원안정성비율_지역기준',
    '재학생충원율_평균_수도권여부기준', '재학생충원율_편차_수도권여부기준', '재학생충원율_표준편차_수도권여부기준', '재학생충원안정성비율_수도권여부기준']
df_merge_part = df_retention[merge_cols]
df_final = pd.merge(df_base, df_merge_part, on=['기준년도', '학교명', '대계열'], how='left')

# 저장
df_final.to_csv('/content/2014~2023_중도탈락_수도권기준_신입충원_재학생안정성포함_충원율포함.csv', index=False, encoding='utf-8-sig')
df_final.head()

In [ ]:
#신입생 충원율 편차 지역기준을 추가안해서 급하게 다시 추가 
df_target = pd.read_csv('/content/2014~2023_중도탈락_수도권기준_신입충원_재학생안정성포함_충원율포함.csv')
df_fresh = pd.read_csv('/content/신입생_충원_현황.csv')

# 신입생 충원율_편차 컬럼 정리 및 병합
df_fresh = df_fresh.rename(columns={'학교': '학교명'})
df_diff = df_fresh[['기준년도', '학교명', '대계열', '충원율_편차']]
df_merged = pd.merge(df_target, df_diff, on=['기준년도', '학교명', '대계열'], how='left')

# 저장 및 확인
df_merged.to_csv('/content/2014~2023_중도탈락_수도권기준_신입충원재학생포함_충원율편차추가.csv', index=False, encoding='utf-8-sig')
df_merged[['학교명', '대계열', '충원율_편차']].head()

In [ ]:
# 새 파생 피처 만들어서 추가
df = pd.read_csv('/content/2014~2023_중도탈락_수도권기준_신입충원재학생포함_충원율편차추가.csv')

# 단위 보정된 1인당 장학금 대비 취업률 계산 이유 : 1인당 장학금 값이 커서 너무 값이 작아짐
# (취업률 ÷ 1인당장학금) × 1,000,000
df['1인당장학금대비취업률'] = (df['취업률(%)'] / df['1인당장학금']) * 1_000_000

# 저장 및 확인
df.to_csv('/content/최종 전처리 파일(final).csv', index=False, encoding='utf-8-sig')
df[['학교명', '기준년도', '취업률(%)', '1인당장학금', '1인당장학금대비취업률']].head()